# Neo RX V1.3.4 — entrenamiento reproducible en Kaggle

1. Adjunta el dataset público **NIH Chest X-rays**.
2. Activa Internet para clonar la etiqueta e instalar TorchXRayVision.
3. Usa primero MODE = pilot y después MODE = full.
4. Selecciona una GPU disponible. Los checkpoints se guardan después de cada época.
5. No añadas secretos, datos clínicos ni archivos .env al notebook.


In [ ]:
from pathlib import Path
import os, subprocess, sys

REPO_URL = "https://github.com/LinoMMJ/NEO-RX.git"
GIT_REF = "v1.3.4"
ROOT = Path("/kaggle/working/neorx")
if not ROOT.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", GIT_REF, REPO_URL, str(ROOT)], check=True)
BACKEND = ROOT / "backend"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(BACKEND / "training/requirements.kaggle.txt")], check=True)
os.chdir(BACKEND)
sys.path.insert(0, str(BACKEND))
print("Código:", ROOT)


In [ ]:
import torch, platform
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Activa Accelerator > GPU antes del piloto o entrenamiento")
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))


In [ ]:
from pathlib import Path
import yaml

input_candidates = []
for child in Path("/kaggle/input").iterdir():
    if any(child.rglob("Data_Entry_2017.csv")):
        input_candidates.append(child)
if len(input_candidates) != 1:
    raise RuntimeError(f"Se esperaba exactamente un dataset NIH adjunto; encontrados: {input_candidates}")
DATA_ROOT = input_candidates[0]
WORK = Path("/kaggle/working")
MANIFESTS = WORK / "manifests"
CHECKPOINTS = WORK / "checkpoints"
RESULTS = WORK / "results"
for directory in (MANIFESTS, CHECKPOINTS, RESULTS):
    directory.mkdir(parents=True, exist_ok=True)

with open(BACKEND / "training/config.kaggle.yaml", encoding="utf-8") as handle:
    config = yaml.safe_load(handle)
config["data"]["data_dir"] = str(DATA_ROOT)
config["data"]["manifest"] = str(MANIFESTS / "clean_manifest.csv")
config["data"]["splits_json"] = str(MANIFESTS / "splits.json")
config["model"]["checkpoint_dir"] = str(CHECKPOINTS)
config["train"]["log_dir"] = str(WORK / "runs")
RUNTIME_CONFIG = WORK / "config.kaggle.runtime.yaml"
with open(RUNTIME_CONFIG, "w", encoding="utf-8") as handle:
    yaml.safe_dump(config, handle, allow_unicode=True, sort_keys=False)
print("Dataset:", DATA_ROOT)
print("Configuración:", RUNTIME_CONFIG)


## Auditoría

La auditoría lee el dataset directamente desde /kaggle/input; no copia los 42 GB a /kaggle/working. Para una comprobación inicial puedes añadir --max-images 2000, pero elimínalo antes del entrenamiento final.


In [ ]:
audit_command = [
    sys.executable, "-m", "training.validate_dataset",
    "--input-root", str(DATA_ROOT),
    "--output-dir", str(MANIFESTS),
    "--workers", "4",
]
if not (MANIFESTS / "clean_manifest.csv").exists():
    subprocess.run(audit_command, check=True)
else:
    print("Se reutiliza el manifiesto existente.")


In [ ]:
subprocess.run([
    sys.executable, "-m", "training.create_splits",
    "--manifest", str(MANIFESTS / "clean_manifest.csv"),
    "--output-dir", str(MANIFESTS),
    "--labels", "pulmonary",
    "--seed", "42",
], check=True)


## Piloto o entrenamiento completo

El piloto usa 2.000 muestras y sirve únicamente para comprobar memoria, tiempos y checkpoints. Antes del entrenamiento completo elimina los checkpoints del piloto o comienza en una salida limpia.


In [ ]:
MODE = "pilot"  # cambia a "full" después de aprobar el piloto
RESUME = None       # ejemplo: "/kaggle/input/neorx-checkpoints/last.pt"

train_command = [
    sys.executable, "-m", "training.train",
    "--config", str(RUNTIME_CONFIG),
    "--device", "cuda",
]
if MODE == "pilot":
    train_command += ["--max-samples", "2000"]
if RESUME:
    train_command += ["--resume", RESUME]
print("Ejecutando:", " ".join(train_command))
subprocess.run(train_command, check=True)


## Evaluación final

Ejecuta esta celda solamente después del entrenamiento completo.


In [ ]:
FINAL = CHECKPOINTS / "finetuned_resnet50_nih.pt"
if MODE != "full":
    print("Evaluación omitida: el piloto no produce métricas finales válidas.")
elif not FINAL.exists():
    raise FileNotFoundError(FINAL)
else:
    subprocess.run([
        sys.executable, "-m", "training.evaluate",
        "--config", str(RUNTIME_CONFIG),
        "--checkpoint", str(FINAL),
        "--device", "cuda",
        "--output-dir", str(RESULTS),
        "--plot",
    ], check=True)


In [ ]:
import shutil
archive = shutil.make_archive("/kaggle/working/neorx-v1.3.4-checkpoints", "zip", "/kaggle/working",
                              base_dir="checkpoints")
print("Descarga:", archive)
print("Conserva también manifests/, results/ y runs/ como outputs del Notebook.")
